In [1]:
"""
SPLIT EXPERIMENT VERSION 4: ~80% Client / ~20% Server
=====================================================
Client: VisionEncoder + TextEncoder + 3xCBAM + 2xTextRefine + text_refine_norm + QAttn + q_gate + q_norm + 2xFusion
Server: 2xFusion + Pool + Head
Client sends: partially-fused visual (B, 49, 256) + partially-fused text (B, 64, 256)
"""
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets", "openpyxl", "tqdm"])
import re
import os, random, time, copy
import numpy as np
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda': print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR = "/kaggle/working/"; os.makedirs(OUTPUT_DIR, exist_ok=True)
SPLIT_NAME = "V4_80pct_client_Adaptive_CAWA"

D = 256; VOCAB_SIZE = 30522; MAX_SEQ = 64; HEADS = 4; DROP = 0.15
CBAM_BLOCKS = 3; TEXT_ENC_LAYERS = 2; TEXT_REFINE_LAYERS = 2; FUSE_LAYERS = 4
NUM_CLIENTS = 5; ROUNDS = 30; BS = 32; SERVER_LR = 3e-4; ENC_LR = 3e-5; WD = 1e-4
FREEZE_ROUNDS = 3

# Number of fusion layers on each side
CLIENT_FUSE_LAYERS = 2
SERVER_FUSE_LAYERS = FUSE_LAYERS - CLIENT_FUSE_LAYERS  # 2

# =============================================================================
# ADAPTIVE CAWA HYPERPARAMETERS
# =============================================================================
CAWA_ALPHA  = 0.05    # Base Reward
CAWA_BETA   = 0.05    # Base Penalty
CAWA_GAMMA  = 0.5     # Exponential streak multiplier
CAWA_LAMBDA = 0.8     # Threshold sensitivity (std devs from mean)
CAWA_TEMP   = 1.0     # Temperature for Softmax-style weight conversion
CAWA_PHI    = 2.0     # Temporal scaling factor (quadratic warmup)

# ─────────────────────────────────────────────────────────────────────
# 3. SLAKE  (mdwiratathya/SLAKE-vqa-english)
# ─────────────────────────────────────────────────────────────────────
# ~14,028 QA pairs (English subset) · 642 images
# Modalities: CT, MRI, X-Ray · Body parts: head/neck/chest/abdomen/pelvis
# Closed-ended (yes/no) + Open-ended (organ, modality, plane, position,
# abnormality, size, color, shape, KG-based questions)

def normalize_answer_slake(ans: str) -> str:
    """Normalize SLAKE answers."""
    ans = ans.strip().lower()
    ans = re.sub(r'[^\w\s\-/.,]', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    # ── Yes / No ──
    yes_set = {'yes', 'yes.', 'yeah', 'yep', 'y', 'correct', 'true'}
    no_set  = {'no', 'no.', 'nope', 'n', 'false', 'incorrect', 'negative',
               'none', 'not sure'}
    if ans in yes_set:
        return 'yes'
    if ans in no_set:
        return 'no'

    # ── Numeric ──
    word_to_num = {'zero': '0', 'one': '1', 'two': '2', 'three': '3',
                   'four': '4', 'five': '5', 'six': '6', 'seven': '7',
                   'eight': '8', 'nine': '9', 'ten': '10'}
    if ans in word_to_num:
        return word_to_num[ans]

    # ── Modality synonyms (SLAKE has CT/MRI/X-Ray) ──
    modality_map = {
        'ct scan': 'ct', 'ct': 'ct', 'computed tomography': 'ct',
        'cat scan': 'ct',
        'mri': 'mri', 'magnetic resonance imaging': 'mri', 'mr': 'mri',
        'mri - Loss contrast': 'mri', 't1': 'mri', 't2': 'mri',
        't1-weighted': 'mri', 't2-weighted': 'mri', 'flair': 'mri',
        'x-ray': 'x-ray', 'x ray': 'x-ray', 'xray': 'x-ray',
        'radiograph': 'x-ray', 'plain film': 'x-ray',
    }
    if ans in modality_map:
        return modality_map[ans]

    # ── Plane synonyms ──
    plane_map = {
        'axial': 'axial', 'transverse': 'axial', 'horizontal': 'axial',
        'axial plane': 'axial', 'transverse plane': 'axial',
        'coronal': 'coronal', 'frontal': 'coronal', 'coronal plane': 'coronal',
        'sagittal': 'sagittal', 'sagittal plane': 'sagittal',
    }
    if ans in plane_map:
        return plane_map[ans]

    # ── Anatomical / organ synonyms (SLAKE covers head/chest/abdomen/pelvis) ──
    anatomy_map = {
        'brain': 'brain', 'cerebral': 'brain', 'cerebrum': 'brain',
        'head': 'brain', 'cranium': 'brain',
        'lung': 'lung', 'lungs': 'lung', 'pulmonary': 'lung',
        'left lung': 'left lung', 'right lung': 'right lung',
        'heart': 'heart', 'cardiac': 'heart',
        'liver': 'liver', 'hepatic': 'liver',
        'kidney': 'kidney', 'kidneys': 'kidney', 'renal': 'kidney',
        'left kidney': 'left kidney', 'right kidney': 'right kidney',
        'spleen': 'spleen', 'splenic': 'spleen',
        'pancreas': 'pancreas', 'pancreatic': 'pancreas',
        'gallbladder': 'gallbladder', 'gall bladder': 'gallbladder',
        'stomach': 'stomach', 'gastric': 'stomach',
        'bladder': 'bladder', 'urinary bladder': 'bladder',
        'spine': 'spine', 'spinal': 'spine', 'vertebral': 'spine',
        'vertebra': 'spine', 'vertebrae': 'spine',
        'chest': 'chest', 'thorax': 'chest', 'thoracic': 'chest',
        'abdomen': 'abdomen', 'abdominal': 'abdomen',
        'pelvis': 'pelvis', 'pelvic': 'pelvis',
        'neck': 'neck', 'cervical': 'neck',
    }
    if ans in anatomy_map:
        return anatomy_map[ans]

    # ── Laterality ──
    lat_map = {
        'right side': 'right', 'right-sided': 'right',
        'left side': 'left', 'left-sided': 'left',
        'both sides': 'bilateral', 'bilateral': 'bilateral', 'both': 'bilateral',
    }
    if ans in lat_map:
        return lat_map[ans]

    # ── Abnormality synonyms ──
    abnorm_map = {
        'normal': 'normal', 'no abnormality': 'normal',
        'no abnormalities': 'normal', 'no finding': 'normal',
        'no findings': 'normal', 'unremarkable': 'normal',
        'tumor': 'tumor', 'tumour': 'tumor', 'mass': 'tumor',
        'neoplasm': 'tumor',
        'inflammation': 'inflammation', 'inflamed': 'inflammation',
        'inflammatory': 'inflammation',
        'fracture': 'fracture', 'broken': 'fracture',
        'effusion': 'effusion', 'fluid': 'effusion',
        'pleural effusion': 'pleural effusion',
        'pneumonia': 'pneumonia',
        'edema': 'edema', 'oedema': 'edema', 'swelling': 'edema',
        'hemorrhage': 'hemorrhage', 'haemorrhage': 'hemorrhage',
        'bleeding': 'hemorrhage',
        'atrophy': 'atrophy', 'atrophic': 'atrophy',
        'calcification': 'calcification', 'calcified': 'calcification',
        'enlarged': 'enlargement', 'enlargement': 'enlargement',
        'hypertrophy': 'enlargement',
    }
    if ans in abnorm_map:
        return abnorm_map[ans]

    # ── Remove articles ──
    ans = re.sub(r'^(the|a|an)\s+', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    return ans


# =============================================================================
# LOAD DATASET INTO RAM
# =============================================================================
from datasets import load_dataset
ds = load_dataset('mdwiratathya/SLAKE-vqa-english')

def extract(sd, name):
    samples = []
    for s in tqdm(sd, desc=name):
        try:
            img = s.get('image'); q = str(s.get('question','')); 
            a = str(s.get('answer','')).strip().lower()
            a = normalize_answer_slake(a)
            if img and q and a:
                samples.append({'image': np.array(img.convert('RGB').resize((224,224)), dtype=np.float32)/255.0, 'question': q, 'answer': a})
        except: continue
    print(f"  {name}: {len(samples)}"); return samples

train_samples = extract(ds['train'], 'train'); test_samples = extract(ds['test'], 'test'); del ds

all_ans = [s['answer'] for s in train_samples + test_samples]
answer_vocab = {'<unk>': 0}
for i, a in enumerate(sorted(set(all_ans))): answer_vocab[a] = i + 1
num_classes = len(answer_vocab)
print(f"  Vocab: {num_classes}, Train: {len(train_samples)}, Test: {len(test_samples)}")

def tokenize(questions):
    ids_l, mask_l = [], []
    for q in questions:
        w = q.lower().split()[:MAX_SEQ-2]
        ids = [1] + [hash(x)%(VOCAB_SIZE-2)+2 for x in w] + [2]
        m = [1.0]*len(ids)
        while len(ids) < MAX_SEQ: ids.append(0); m.append(0.0)
        ids_l.append(ids[:MAX_SEQ]); mask_l.append(m[:MAX_SEQ])
    return ids_l, mask_l

class VQADataset(Dataset):
    def __init__(self, samples, vocab, augment=False):
        self.samples=samples; self.vocab=vocab; self.augment=augment
        self.ids, self.masks = tokenize([s['question'] for s in samples])
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]; img = torch.tensor(s['image']).permute(2,0,1)
        if self.augment and random.random()>0.5: img = img.flip(-1)
        return img, torch.tensor(self.ids[idx], dtype=torch.long), torch.tensor(self.masks[idx], dtype=torch.float32), self.vocab.get(s['answer'],0)

def collate_fn(batch):
    imgs,ids,masks,lbls = zip(*batch)
    return torch.stack(imgs), torch.stack(ids), torch.stack(masks), torch.tensor(lbls, dtype=torch.long)

idx = np.random.permutation(len(train_samples)); sz = len(train_samples)//NUM_CLIENTS
client_splits = {c: idx[c*sz:(c+1)*sz if c<NUM_CLIENTS-1 else len(train_samples)].tolist() for c in range(NUM_CLIENTS)}
client_loaders = {}
for cid, indices in client_splits.items():
    client_loaders[cid] = DataLoader(VQADataset([train_samples[i] for i in indices], answer_vocab, augment=True),
                                      batch_size=BS, shuffle=True, num_workers=2, pin_memory=True, collate_fn=collate_fn)
    print(f"  Client {cid}: {len(indices)} samples")

test_loader = DataLoader(VQADataset(test_samples, answer_vocab), batch_size=BS, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

# =============================================================================
# MODEL BLOCKS — SAME as centralized (copy-paste identical)
# =============================================================================
class TransformerBlock(nn.Module):
    def __init__(self, dim, n_heads=4, ffn_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1=nn.LayerNorm(dim); self.norm2=nn.LayerNorm(dim)
        self.attn=nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.ffn=nn.Sequential(nn.Linear(dim,dim*ffn_ratio), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim*ffn_ratio,dim), nn.Dropout(dropout))
    def forward(self, x, mask=None):
        h=self.norm1(x); kpm=(mask==0) if mask is not None else None
        h,_=self.attn(h,h,h,key_padding_mask=kpm); x=x+h; return x+self.ffn(self.norm2(x))

class VisionEncoder(nn.Module):
    def __init__(self, dim=256):
        super().__init__()
        self.conv1=nn.Conv2d(3,32,7,stride=2,padding=3,bias=False); self.bn1=nn.BatchNorm2d(32)
        self.pool1=nn.MaxPool2d(3,stride=2,padding=1)
        self.conv2=nn.Conv2d(32,64,3,stride=2,padding=1,bias=False); self.bn2=nn.BatchNorm2d(64)
        self.conv3=nn.Conv2d(64,128,3,stride=2,padding=1,bias=False); self.bn3=nn.BatchNorm2d(128)
        self.conv4=nn.Conv2d(128,dim,3,stride=2,padding=1,bias=False); self.bn4=nn.BatchNorm2d(dim)
        self.norm=nn.LayerNorm(dim)
    def forward(self, x):
        h=self.pool1(F.silu(self.bn1(self.conv1(x)))); h=F.silu(self.bn2(self.conv2(h)))
        h=F.silu(self.bn3(self.conv3(h))); h=F.silu(self.bn4(self.conv4(h)))
        B,C,H,W=h.shape; return self.norm(h.permute(0,2,3,1).reshape(B,H*W,C))

class TextEncoder(nn.Module):
    def __init__(self, vocab_size=30522, dim=256, n_layers=2, n_heads=4, max_len=64, dropout=0.1):
        super().__init__()
        self.tok_embed=nn.Embedding(vocab_size,dim); self.pos_embed=nn.Parameter(torch.randn(1,max_len,dim)*0.02)
        self.embed_norm=nn.LayerNorm(dim); self.embed_drop=nn.Dropout(dropout)
        self.blocks=nn.ModuleList([TransformerBlock(dim,n_heads,dropout=dropout) for _ in range(n_layers)])
        self.final_norm=nn.LayerNorm(dim)
    def forward(self, input_ids, mask=None):
        L=input_ids.shape[1]; x=self.tok_embed(input_ids)+self.pos_embed[:,:L,:]
        x=self.embed_drop(self.embed_norm(x))
        for blk in self.blocks: x=blk(x,mask=mask)
        return self.final_norm(x)

class ChannelAttention(nn.Module):
    def __init__(self, ch, ratio=8):
        super().__init__(); self.fc1=nn.Linear(ch,ch//ratio,bias=False); self.fc2=nn.Linear(ch//ratio,ch,bias=False)
    def forward(self, x):
        avg=x.mean(dim=[1,2],keepdim=True); mx=x.amax(dim=[1,2],keepdim=True)
        return x*torch.sigmoid(self.fc2(F.silu(self.fc1(avg)))+self.fc2(F.silu(self.fc1(mx))))

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__(); self.conv1=nn.Conv2d(2,8,3,padding=1,bias=False); self.conv2=nn.Conv2d(2,8,3,padding=2,dilation=2,bias=False)
        self.fuse=nn.Conv2d(16,1,1,bias=False)
    def forward(self, x):
        xp=x.permute(0,3,1,2); avg=xp.mean(1,keepdim=True); mx=xp.amax(1,keepdim=True)
        cat=torch.cat([avg,mx],1); ms=torch.cat([self.conv1(cat),self.conv2(cat)],1)
        return x*torch.sigmoid(self.fuse(ms)).permute(0,2,3,1)

class CBAMBlock(nn.Module):
    def __init__(self, ch):
        super().__init__(); self.ca=ChannelAttention(ch); self.sa=SpatialAttention()
        self.ffn=nn.Sequential(nn.Linear(ch,ch*2),nn.GELU(),nn.Linear(ch*2,ch))
        self.norm1=nn.LayerNorm(ch); self.norm2=nn.LayerNorm(ch)
    def forward(self, tokens):
        B,N,C=tokens.shape; sp=tokens.reshape(B,7,7,C); sp=self.sa(self.ca(sp))
        tokens=self.norm1(tokens+sp.reshape(B,N,C)); return self.norm2(tokens+self.ffn(tokens))

class FusionLayer(nn.Module):
    def __init__(self, dim, n_heads, dropout):
        super().__init__()
        self.v2t=nn.MultiheadAttention(dim,n_heads,dropout=dropout,batch_first=True); self.v2t_norm=nn.LayerNorm(dim)
        self.v2t_ffn=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Dropout(dropout),nn.Linear(dim*4,dim)); self.v2t_ffn_norm=nn.LayerNorm(dim)
        self.t2v=nn.MultiheadAttention(dim,n_heads,dropout=dropout,batch_first=True); self.t2v_norm=nn.LayerNorm(dim)
        self.t2v_ffn=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Dropout(dropout),nn.Linear(dim*4,dim)); self.t2v_ffn_norm=nn.LayerNorm(dim)
    def forward(self, v, t, text_kpm=None):
        o,_=self.v2t(v,t,t,key_padding_mask=text_kpm); v=self.v2t_norm(v+o); v=self.v2t_ffn_norm(v+self.v2t_ffn(v))
        o,_=self.t2v(t,v,v); t=self.t2v_norm(t+o); t=self.t2v_ffn_norm(t+self.t2v_ffn(t)); return v,t

# =============================================================================
# SPLIT V4: Client = VisionEnc + TextEnc + CBAM + TextRefine + QAttn + 2xFusion (~80%)
#            Server = 2xFusion + Pool + Head (~20%)
# =============================================================================
class ClientEncoder(nn.Module):
    """Client portion: VisionEnc + TextEnc + CBAM + TextRefine + QAttn + 2xFusion (~80% of total params)."""
    def __init__(self):
        super().__init__()
        self.vision_enc = VisionEncoder(D)
        self.text_enc = TextEncoder(VOCAB_SIZE, D, TEXT_ENC_LAYERS, HEADS, MAX_SEQ, DROP)
        self.vis_refine = nn.ModuleList([CBAMBlock(D) for _ in range(CBAM_BLOCKS)])
        self.text_refine = nn.ModuleList([TransformerBlock(D, HEADS, dropout=DROP) for _ in range(TEXT_REFINE_LAYERS)])
        self.text_refine_norm = nn.LayerNorm(D)
        # Q-Attention on client
        self.q_attn = nn.MultiheadAttention(D, HEADS, dropout=DROP, batch_first=True)
        self.q_gate = nn.Linear(D, D); self.q_norm = nn.LayerNorm(D)
        # First 2 fusion layers on client
        self.fusion_layers = nn.ModuleList([FusionLayer(D, HEADS, DROP) for _ in range(CLIENT_FUSE_LAYERS)])
        self.frozen = True; self._set_frozen(True)
    def _set_frozen(self, f):
        self.frozen = f
        for p in self.parameters(): p.requires_grad = not f
    def unfreeze(self):
        self._set_frozen(False)
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"    Client unfrozen: {n:,} ({n/1e6:.1f}M)")
    def freeze(self): self._set_frozen(True)
    def encode(self, images, input_ids, text_mask):
        v = self.vision_enc(images)
        t = self.text_enc(input_ids, mask=text_mask)
        for blk in self.vis_refine: v = blk(v)
        for blk in self.text_refine: t = blk(t, mask=text_mask)
        t = self.text_refine_norm(t)
        # Q-Attention
        q_cls = t[:,0:1,:].expand(-1,v.shape[1],-1)
        ao,_ = self.q_attn(q_cls, v, v); gate = torch.sigmoid(self.q_gate(ao))
        v = self.q_norm(v + v*gate + ao*(1-gate))
        # First 2 fusion layers
        kpm = (text_mask == 0)
        for fl in self.fusion_layers: v, t = fl(v, t, text_kpm=kpm)
        return v, t

class ServerModel(nn.Module):
    """Server portion: 2xFusion + Pool + Head (~20%)."""
    def __init__(self, num_classes):
        super().__init__()
        # Remaining 2 fusion layers
        self.fusion_layers = nn.ModuleList([FusionLayer(D, HEADS, DROP) for _ in range(SERVER_FUSE_LAYERS)])
        self.pool_query = nn.Parameter(torch.randn(1,1,D)*0.02)
        self.pool_attn = nn.MultiheadAttention(D, HEADS, dropout=DROP, batch_first=True); self.pool_norm = nn.LayerNorm(D)
        self.head_fc1 = nn.Linear(D,D); self.head_drop1 = nn.Dropout(DROP)
        self.head_fc2 = nn.Linear(D,D//2); self.head_drop2 = nn.Dropout(DROP)
        self.head_out = nn.Linear(D//2, num_classes); self.head_res = nn.Linear(D, D//2); self.head_norm = nn.LayerNorm(D//2)

    def forward(self, v, t, text_mask):
        kpm = (text_mask == 0)
        for fl in self.fusion_layers: v, t = fl(v, t, text_kpm=kpm)
        combined = torch.cat([v,t], dim=1); B = combined.shape[0]
        pq = self.pool_query.expand(B,-1,-1)
        pooled,_ = self.pool_attn(pq, combined, combined)
        fused = self.pool_norm(pq + pooled).squeeze(1)
        h = self.head_drop1(F.gelu(self.head_fc1(fused)))
        h = self.head_drop2(F.gelu(self.head_fc2(h)))
        return self.head_out(self.head_norm(h + self.head_res(fused)))

client_enc = ClientEncoder().to(device)
server_model = ServerModel(num_classes).to(device)

n_client = sum(p.numel() for p in client_enc.parameters())
n_server = sum(p.numel() for p in server_model.parameters())
n_total = n_client + n_server
print(f"\n  [{SPLIT_NAME}] Client: {n_client:,} ({n_client/1e6:.1f}M) = {100*n_client/n_total:.1f}%")
print(f"  [{SPLIT_NAME}] Server: {n_server:,} ({n_server/1e6:.1f}M) = {100*n_server/n_total:.1f}%")
print(f"  [{SPLIT_NAME}] Total:  {n_total:,} ({n_total/1e6:.1f}M)")

# =============================================================================
# ADAPTIVE CAWA UPDATE LOGIC
# =============================================================================
def grad_cos_sim(grads, idx, cids, current_reputations, temp):
    if len(grads) < 2: return 0.5
    
    # Flatten grads for each client
    flat_norm = []
    for gl in grads:
        flat = np.concatenate([g.numpy().ravel() for g in gl])
        n = np.linalg.norm(flat)
        flat_norm.append(flat / n if n > 1e-8 else flat * 0.0)
        
    c_target = flat_norm[idx]
    sims, weights = [], []
    max_rep = max(current_reputations.values())
    
    for i in range(len(flat_norm)):
        if i != idx:
            sim = np.dot(c_target, flat_norm[i])
            sims.append(sim)
            w = np.exp((current_reputations[cids[i]] - max_rep) / temp)
            weights.append(w)
            
    w_sum = sum(weights) + 1e-8
    weights = [w / w_sum for w in weights]
    
    weighted_avg_sim = np.average(sims, weights=weights)
    return float(weighted_avg_sim)

# =============================================================================
# TRAINING
# =============================================================================
print("\n" + "="*60 + f"\nTRAINING ({SPLIT_NAME}: U-Shaped Split + Adaptive CAWA)\n" + "="*60)
criterion = nn.CrossEntropyLoss()
server_opt = torch.optim.AdamW(server_model.parameters(), lr=SERVER_LR, weight_decay=WD)
enc_opts = {}

# Adaptive CAWA Tracking
reputation = {c: 0.0 for c in range(NUM_CLIENTS)}
streaks = {c: 0 for c in range(NUM_CLIENTS)}

history = {'round':[],'avg_train_loss':[],'avg_train_acc':[],'test_loss':[],'test_acc':[],'round_time':[]}
for cid in range(NUM_CLIENTS):
    history[f'c{cid}_loss']=[]
    history[f'c{cid}_acc']=[]
    history[f'c{cid}_reputation']=[]
    history[f'c{cid}_weight']=[]
    history[f'c{cid}_similarity']=[]
    history[f'c{cid}_time']=[]

best_acc, best_state, unfrozen = 0.0, None, False

@torch.no_grad()
def eval_usplit(loader):
    server_model.eval(); was=client_enc.frozen; client_enc.freeze()
    ls,c,t = 0.0,0,0
    for imgs,ids,masks,lbls in loader:
        imgs,ids,masks,lbls = imgs.to(device),ids.to(device),masks.to(device),lbls.to(device)
        vis, txt = client_enc.encode(imgs, ids, masks)
        logits = server_model(vis, txt, masks)
        ls += criterion(logits,lbls).item()*lbls.size(0); c += (logits.argmax(-1)==lbls).sum().item(); t += lbls.size(0)
    if not was: client_enc.unfreeze()
    return ls/t, 100*c/t

for rnd in range(1, ROUNDS+1):
    t0 = time.time()
    if rnd == FREEZE_ROUNDS+1 and not unfrozen:
        print(f"\n  === Phase 2: Unfreezing client (round {rnd}) ===")
        client_enc.unfreeze(); unfrozen = True
        for cid in range(NUM_CLIENTS):
            enc_opts[cid] = torch.optim.AdamW(client_enc.parameters(), lr=ENC_LR, weight_decay=WD)
        print()

    server_model.train()
    rnd_grads, rnd_cids, rnd_losses = [], [], []
    rnd_correct, rnd_total = 0, 0

    pbar = tqdm(range(NUM_CLIENTS), desc=f"R{rnd:02d}/{ROUNDS}", leave=False)
    for cid in pbar:
        c_t0 = time.time()
        cl, cc, ct = 0.0, 0, 0
        c_grad_sum, num_batches = None, 0

        # ADAPTIVE CAWA: Calculate bounded [0, 1] weight from unbounded reputation
        max_rep = max(reputation.values()) if reputation else 0.0
        trust_weight = np.exp((reputation[cid] - max_rep) / CAWA_TEMP)
        history[f'c{cid}_weight'].append(round(trust_weight, 3))

        for imgs, ids, masks, lbls in client_loaders[cid]:
            imgs, ids, masks, lbls = imgs.to(device), ids.to(device), masks.to(device), lbls.to(device)

            if unfrozen:
                vis, txt = client_enc.encode(imgs, ids, masks)
                server_opt.zero_grad()
                logits = server_model(vis, txt, masks)
                
                raw_loss = criterion(logits, lbls)
                weighted_loss = raw_loss * trust_weight
                weighted_loss.backward()

                if c_grad_sum is None:
                    c_grad_sum = [p.grad.detach().cpu().clone() for p in server_model.parameters() if p.grad is not None]
                else:
                    grad_idx = 0
                    for p in server_model.parameters():
                        if p.grad is not None:
                            c_grad_sum[grad_idx] += p.grad.detach().cpu()
                            grad_idx += 1
                num_batches += 1

                nn.utils.clip_grad_norm_(server_model.parameters(), 1.0); server_opt.step()
                nn.utils.clip_grad_norm_(client_enc.parameters(), 1.0)
                enc_opts[cid].step(); enc_opts[cid].zero_grad()
            else:
                with torch.no_grad(): vis, txt = client_enc.encode(imgs, ids, masks)
                server_opt.zero_grad()
                logits = server_model(vis, txt, masks)
                
                raw_loss = criterion(logits, lbls)
                weighted_loss = raw_loss * trust_weight
                weighted_loss.backward()

                if c_grad_sum is None:
                    c_grad_sum = [p.grad.detach().cpu().clone() for p in server_model.parameters() if p.grad is not None]
                else:
                    grad_idx = 0
                    for p in server_model.parameters():
                        if p.grad is not None:
                            c_grad_sum[grad_idx] += p.grad.detach().cpu()
                            grad_idx += 1
                num_batches += 1

                nn.utils.clip_grad_norm_(server_model.parameters(), 1.0); server_opt.step()

            cl += raw_loss.item()*lbls.size(0); cc += (logits.argmax(-1)==lbls).sum().item(); ct += lbls.size(0)

        c_time = time.time() - c_t0

        if c_grad_sum is not None:
            rnd_grads.append([g / num_batches for g in c_grad_sum])
            rnd_cids.append(cid)

        c_l=cl/max(ct,1); c_a=100*cc/max(ct,1); rnd_losses.append(c_l); rnd_correct+=cc; rnd_total+=ct
        history[f'c{cid}_loss'].append(round(c_l,4))
        history[f'c{cid}_acc'].append(round(c_a,2))
        history[f'c{cid}_time'].append(round(c_time,2))
        pbar.set_postfix(C=cid, l=f"{c_l:.3f}", a=f"{c_a:.1f}%", t=f"{c_time:.1f}s")

    # =========================================================================
    # ROUND END: ADAPTIVE SCORING & UNBOUNDED UPDATES WITH TEMPORAL SCALING
    # =========================================================================
    similarities = {}
    if rnd_grads:
        for ii, ci in enumerate(rnd_cids):
            sim = grad_cos_sim(rnd_grads, ii, rnd_cids, reputation, CAWA_TEMP)
            similarities[ci] = sim
            
        sim_values = list(similarities.values())
        mu = np.mean(sim_values)
        sigma = np.std(sim_values)
        tau_pos = mu + (CAWA_LAMBDA * sigma)
        tau_neg = mu - (CAWA_LAMBDA * sigma)
        
        rho = (rnd / ROUNDS) ** CAWA_PHI
        
        for ci in rnd_cids:
            sim = similarities[ci]
            if sim > tau_pos:
                streaks[ci] = max(1, streaks[ci] + 1)
                reputation[ci] += rho * CAWA_ALPHA * np.exp(CAWA_GAMMA * (streaks[ci] - 1))
            elif sim < tau_neg:
                streaks[ci] = min(-1, streaks[ci] - 1)
                reputation[ci] -= rho * CAWA_BETA * np.exp(CAWA_GAMMA * (abs(streaks[ci]) - 1))
            else:
                streaks[ci] = 0

    for cid in range(NUM_CLIENTS):
        history[f'c{cid}_reputation'].append(round(reputation[cid], 3))
        history[f'c{cid}_similarity'].append(round(similarities.get(cid, 0.0), 4))

    te_l, te_a = eval_usplit(test_loader)
    rt=time.time()-t0; avg_l=np.mean(rnd_losses); avg_a=100*rnd_correct/max(rnd_total,1)
    history['round'].append(rnd); history['avg_train_loss'].append(round(avg_l,4))
    history['avg_train_acc'].append(round(avg_a,2)); history['test_loss'].append(round(te_l,4))
    history['test_acc'].append(round(te_a,2)); history['round_time'].append(round(rt,1))

    mk=""
    if te_a>best_acc: best_acc=te_a; best_state=copy.deepcopy(server_model.state_dict()); mk=" ★"
    ph="P2" if unfrozen else "P1"
    sc=" ".join(f"C{c}:{np.exp((reputation[c]-max(reputation.values()))/CAWA_TEMP):.2f}" for c in range(NUM_CLIENTS))
    print(f"R{rnd:02d} [{rt:.1f}s] [{ph}]  Train: {avg_l:.4f}/{avg_a:.1f}%  Test: {te_l:.4f}/{te_a:.1f}%{mk}")
    print(f"  Weights: {sc}")

if best_state: server_model.load_state_dict(best_state)
te_l, te_a = eval_usplit(test_loader)
print(f"\n{'='*60}\n[{SPLIT_NAME}] FINAL: {te_a:.2f}% (best: {best_acc:.2f}%)\n{'='*60}")

# =============================================================================
# SAVE EXCEL
# =============================================================================
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment

wb=openpyxl.Workbook(); ws=wb.active; ws.title="Training"
hf=Font(name='Arial',bold=True,size=11,color='FFFFFF'); hfi=PatternFill(start_color='1A5276',end_color='1A5276',fill_type='solid')

headers=['Round','Avg Train Loss','Avg Train Acc (%)','Test Loss','Test Acc (%)','Time (s)']
for cid in range(NUM_CLIENTS):
    headers+=[f'C{cid} Loss', f'C{cid} Acc (%)', f'C{cid} Reputation', f'C{cid} Applied Weight', f'C{cid} Similarity', f'C{cid} Time (s)']

for c,h in enumerate(headers,1):
    cl=ws.cell(row=1,column=c,value=h); cl.font=hf; cl.fill=hfi; cl.alignment=Alignment(horizontal='center')

for i,rnd in enumerate(history['round']):
    r=i+2; ws.cell(row=r,column=1,value=rnd)
    ws.cell(row=r,column=2,value=history['avg_train_loss'][i]); ws.cell(row=r,column=3,value=history['avg_train_acc'][i])
    ws.cell(row=r,column=4,value=history['test_loss'][i]); ws.cell(row=r,column=5,value=history['test_acc'][i])
    ws.cell(row=r,column=6,value=history['round_time'][i])
    
    # 6 Base columns. Each client has 6 columns.
    for cid in range(NUM_CLIENTS):
        base_col = 7 + (cid * 6)
        ws.cell(row=r,column=base_col,value=history[f'c{cid}_loss'][i])
        ws.cell(row=r,column=base_col+1,value=history[f'c{cid}_acc'][i])
        ws.cell(row=r,column=base_col+2,value=history[f'c{cid}_reputation'][i])
        ws.cell(row=r,column=base_col+3,value=history[f'c{cid}_weight'][i])
        ws.cell(row=r,column=base_col+4,value=history[f'c{cid}_similarity'][i])
        ws.cell(row=r,column=base_col+5,value=history[f'c{cid}_time'][i])

ws2=wb.create_sheet("Summary")
for i,(k,v) in enumerate([
    ("Split",SPLIT_NAME),("Method","U-Shaped Split + Adaptive CAWA"),("Dataset","VQA-RAD"),
    ("Client portion","VisionEnc + TextEnc + 3xCBAM + 2xTextRefine + QAttn + 2xFusion"),
    ("Client params",f"{n_client:,} ({100*n_client/n_total:.1f}%)"),
    ("Server params",f"{n_server:,} ({100*n_server/n_total:.1f}%)"),
    ("Total",f"{n_total:,}"),("Classes",num_classes),
    ("Clients",NUM_CLIENTS),("Rounds",ROUNDS),("Freeze Rounds",FREEZE_ROUNDS),
    ("Server LR",SERVER_LR),("Encoder LR",ENC_LR),
    ("Best Test",round(best_acc,2)),("Final Test",round(te_a,2))], 1):
    ws2.cell(row=i,column=1,value=k).font=Font(bold=True,name='Arial'); ws2.cell(row=i,column=2,value=v)

for s in [ws,ws2]:
    for col in s.columns: s.column_dimensions[col[0].column_letter].width=max(len(str(c.value or '')) for c in col)+2

p=f"{OUTPUT_DIR}/slake_{SPLIT_NAME}_results.xlsx"; wb.save(p); print(f"\nSaved → {p}\nDONE!")

Device: cuda
GPU: Tesla T4, VRAM: 15.6 GB


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/31.1M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/12.2M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/8.34M [00:00<?, ?B/s]

...

    Client unfrozen: 15,714,224 (15.7M)
R28 [17.2s] [P2]  Train: 0.1599/94.7%  Test: 1.5099/67.9%
  Weights: C0:0.19 C1:0.12 C2:0.09 C3:0.10 C4:1.00


R29/30:   0%|          | 0/5 [00:00<?, ?it/s]

    Client unfrozen: 15,714,224 (15.7M)
R29 [17.4s] [P2]  Train: 0.1495/95.5%  Test: 1.5292/68.7%
  Weights: C0:0.20 C1:0.12 C2:0.09 C3:0.10 C4:1.00


R30/30:   0%|          | 0/5 [00:00<?, ?it/s]

    Client unfrozen: 15,714,224 (15.7M)
R30 [17.2s] [P2]  Train: 0.1596/95.1%  Test: 1.5432/68.0%
  Weights: C0:0.21 C1:0.10 C2:0.09 C3:0.10 C4:1.00
    Client unfrozen: 15,714,224 (15.7M)

[V4_80pct_client_Adaptive_CAWA] FINAL: 68.14% (best: 69.65%)

Saved → /kaggle/working//slake_V4_80pct_client_Adaptive_CAWA_results.xlsx
DONE!
